# Faster indicators - port call detection and reporting

Stakeholder notebook.

In [1]:
!pip install pandera

In [2]:
# import sedona.register  # removed for Spark Connect
import pandas as pd
import cloudpickle
import pickle
import warnings
import subprocess
import sys
from getpass import getpass


from operator import itemgetter
from functools import partial
from io import StringIO
from ipywidgets import HBox, VBox
from pyspark.sql.functions import col, broadcast

from IPython.display import display, HTML

from shipping_indicators import ports, ships, positions, ui, utils, reporting, livechecks
from shipping_indicators.port_calls.speed_spark import identify_port_call_candidates, dedup_port_calls, column_sets
from shipping_indicators.port_calls.common_spark import check_vessel_in_port_zone, redact_startend_times
from shipping_indicators.port_calls import common_pandas as pc_common_pandas
from shipping_indicators.port_calls import common_spark as pc_common_spark

In [3]:
import sys
import os

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

In [4]:
# Temp fix - build spark session not being automatically initialized
from pyspark.sql import SparkSession
spark = (
    SparkSession
    .builder
    .appName('SparkApp')
    .getOrCreate()
)

In [5]:
# Register Apache Sedona, provides PostGIS compatible geospatial querying in Spark
# assert sedona.register.SedonaRegistrator.registerAll(spark)  # type: ignore  # removed for Spark Connect

## Configuration

In [6]:
user = 'onsfi'
country_code = 'GB'
statcode5 = 'A'
mangle = False

# activate the following (set to True) to input manual start and end dates for the indicator 
manual_date_definition = False
start_date_manual = '2025-12-01'
end_date_manual = '2025-12-14'

In [7]:
if manual_date_definition == True:
    start_date_picker, end_date_picker = ui.create_date_pickers(start=start_date_manual, end=end_date_manual)
else:
    start_date_picker, end_date_picker = ui.create_date_pickers()

sr_map_selector = ui.ship_register_map_dropdown(maps=ships.list_ship_register_maps())
selector = VBox([HBox([start_date_picker, end_date_picker]), sr_map_selector])
selector

In [8]:
reporting_start_date, ais_start_date, end_date = (
    itemgetter("reporting_start_date", "ais_start_date", "end_date")
    (ui.get_date_selections(start_date_picker, end_date_picker))
)
mmsi_sr_map = sr_map_selector.value

# Master ship register — maintained by Saurabh (user-sc5k-ochag).
# To update: change the date below to match the new snapshot filename.
# Spark reads the full partitioned Parquet directory directly.
MASTER_SR_PATH = "s3a://datalab-142496269814-user-bucket/user-sc5k-ochag/ship_register/mmsi_ship_register_2026-06-01.parquet"

print(f'Port Calls date range:\t{ais_start_date} - {end_date}')
print(f'Reporting date range:\t{reporting_start_date} - {end_date}')
print(f'Ship register (master):\t{MASTER_SR_PATH}')

Port Calls date range:	2025-11-24 - 2025-12-14
Reporting date range:	2025-12-01 - 2025-12-14
AIS to SR mapping:	mmsi_ship_register_2023-10-30.parquet


In [9]:
display(HTML("<style>.container { width:85% !important; }</style>"))
pd.set_option("display.max_columns", 101)
logger = utils.setup_logging()
warnings.simplefilter(action='ignore', category=FutureWarning)

## Data Sources

### Ship Register

In [10]:
t_calc_start = pd.Timestamp.now()

logger.info(f'Loading ship data from {MASTER_SR_PATH}')
ship_register_sdf = ships.load_mmsi_to_ship_register_mapping(spark, MASTER_SR_PATH)

logger.info(f'Ship Register vessel count: {ship_register_sdf.select("LRIMOShipNo").distinct().count()}')
logger.info(f'Ship Register vessel identifier map size: {ship_register_sdf.count()}')

ships_sdf = (
    ships.select_ships_by_statcode5(ship_register_sdf, statcode5)
    .select('mmsi', 'MaritimeMobileServiceIdentityMMSINumber', 'LRIMOShipNo', 'StatCode5')
)

logger.info(f'Selected (trading) ships vessel_count: {ships_sdf.select("LRIMOShipNo").distinct().count()}')

2026-03-10 11:00:08,478 - ungp_port_calls - INFO - Loading ship data from mmsi_ship_register_2023-10-30.parquet
2026-03-10 11:00:09,630 - ungp_port_calls - INFO - Ship Register vessel count: 104768
2026-03-10 11:00:10,013 - ungp_port_calls - INFO - Ship Register vessel identifier map size: 120373
2026-03-10 11:00:10,682 - ungp_port_calls - INFO - Selected (trading) ships vessel_count: 54054


### Port Zones

In [11]:
port_zones_gdf = ports.load_from_package()
gb_port_zone_count = (port_zones_gdf.locode.str[:2] == 'GB').sum()
logger.info(f'Port zone count (Global): {len(port_zones_gdf)}')
logger.info(f'Port zone count (GB): {gb_port_zone_count}')

port_zones_sdf = spark.createDataFrame(port_zones_gdf.astype({"port_zone": str}))
ports_knn = port_zones_gdf.pipe(ports.explode_port_geometry).pipe(ports.fit_knn)

2026-03-10 11:00:10,759 - ungp_port_calls - INFO - Port zone count (Global): 2979
2026-03-10 11:00:10,760 - ungp_port_calls - INFO - Port zone count (GB): 330
/opt/python/lib/python3.11/site-packages/sklearn/neighbors/_base.py:501: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


### AIS position data

AIS position data is uploaded weekly (on a Monday), the data is stored as parquet files partitioned by geographic region (using Uber's [H3](https://eng.uber.com/h3/)).

For Faster Indicators reporting the following filters are used:
* Geographic: Uber H3 regions local to the British Isles
* Vessel type: Trading vessels only

In [12]:
uk_ais_h3_filter = partial(positions.ais_h3_filter_by_locode, port_zones_gdf=port_zones_gdf, locodes=['GBFXT', 'GBLER', 'NLRTM'])

ais_df = (
    positions.load_ais_global_df(spark, ais_start_date, end_date)
    .transform(uk_ais_h3_filter)
    .drop('H3Index_0', 'message_type')
    .join(ships_sdf, on='mmsi', how='inner')
    .transform(positions.replace_ais_ship_identifiers)
    .transform(positions.fix_ais_data_types)
)

t_aisdf_done = pd.Timestamp.now()

## Identify Port Calls

In [13]:
#reload(pc_common_pandas)
#reload(pc_common_spark)

cloudpickle.register_pickle_by_value(pc_common_pandas)
cloudpickle.register_pickle_by_value(pc_common_spark)

find_nearest_port = partial(pickle.loads(cloudpickle.dumps(pc_common_spark.find_nearest_port)), ports_knn=ports_knn)
add_haversine_distance = pickle.loads(cloudpickle.dumps(pc_common_spark.add_haversine_distance))

port_calls_sdf = (
    ais_df
    .transform(identify_port_call_candidates)
    .transform(find_nearest_port)
    .join(port_zones_sdf, on="locode", how="left")
    .transform(add_haversine_distance)
    .transform(check_vessel_in_port_zone)
    .transform(dedup_port_calls)
    .transform(redact_startend_times)
    .dropna(subset='time_arrival')
    .join(broadcast(ship_register_sdf.drop('StatCode5', 'vessel_name')), on='mmsi', how='inner')
    .orderBy("imo", col("time_arrival").desc())
    .select(*column_sets['reporting'])
    .cache()
)

ct_port_calls = port_calls_sdf.count()
logger.info(f'Port visits identified: {ct_port_calls}')
t_portcalls_done = pd.Timestamp.now()

2026-03-10 11:00:57,004 - ungp_port_calls - INFO - Port visits identified: 125419


## Vessel filtering

Removes high frequency, short journey ferries

In [14]:
vessels_active_sdf = reporting.identify_active_vessels_for_country(port_calls_sdf, country_code)
ct_vessels_active = vessels_active_sdf.count()
logger.info(f'Identified {ct_vessels_active} vessels visiting {country_code} ports')

port_calls_by_active_vessels_df = reporting.get_port_calls_by_active_vessels(port_calls_sdf, vessels_active_sdf, port_zones_sdf)
ct_port_calls_by_active_vessels = port_calls_by_active_vessels_df.imo.count()
logger.info(f'Found {ct_port_calls_by_active_vessels} port calls by vessels active in {country_code} ports')

vessel_activity_df = reporting.summarise_vessel_activity(port_calls_by_active_vessels_df)
ct_vessels_active_gb = len(vessel_activity_df)
logger.info(f'Summarised activity for {ct_vessels_active_gb} vessels active in {country_code} ports')

vessels_excluded_df = reporting.select_excluded_vessels(vessel_activity_df)
excluded_imos = vessels_excluded_df.imo.to_list()
ct_excluded_imos = len(excluded_imos)
logger.info(f'Excluding {ct_excluded_imos} vessels')
display(utils.create_download_link(vessels_excluded_df, title='Excluded Vessels', filename='vessels_excluded.csv'))
vessels_excluded_df.groupby(['StatCode5', 'ShiptypeLevel5']).size().sort_values(ascending=False)

2026-03-10 11:01:00,494 - ungp_port_calls - INFO - Identified 1306 vessels visiting GB ports
2026-03-10 11:01:06,843 - ungp_port_calls - INFO - Found 22111 port calls by vessels active in GB ports
2026-03-10 11:02:29,857 - ungp_port_calls - INFO - Summarised activity for 1177 vessels active in GB ports
2026-03-10 11:02:29,876 - ungp_port_calls - INFO - Excluding 36 vessels


StatCode5  ShiptypeLevel5                 
A36A2PR    Passenger/Ro-Ro Ship (Vehicles)    18
A37B2PS    Passenger Ship                     13
A36B2PL    Passenger/Landing Craft             2
A31A2GX    General Cargo Ship                  1
A31C2GD    Deck Cargo Ship                     1
A35A2RR    Ro-Ro Cargo Ship                    1
dtype: int64

## Reporting

In [15]:
# Ship Register ShiptypeLevel5 to Faster Indicators ship type mapping
fi_ship_types_map_df = reporting.load_ship_type_mapping()
ct_fi_ship_types_map = len(fi_ship_types_map_df)
logger.info(f'Ship Register vessel type to Faster Indicators simplified reporting vessel type mappings: {ct_fi_ship_types_map}')

# Faster Indicators port shapes effectively group smaller "sub-ports" together into "super-ports"
fi_locode_map_df = reporting.load_fi_locode_mapping().rename(columns={'fi_locode': 'reporting_locode'})
ct_fi_locode_map = len(fi_locode_map_df)
logger.info(f'UN Locode to Faster Indicators simplified reporting (super port) locode mappings: {ct_fi_locode_map}')

# Faster Indicators ports to be reported
fi_port_locodes = ports.load_from_package(file="ports_fi_boxes.csv").locode.to_list()
ct_fi_port_locodes = len(fi_port_locodes)
logger.info(f'Faster Indicators reporting port zone count: {ct_fi_port_locodes}')

2026-03-10 11:02:29,904 - ungp_port_calls - INFO - Ship Register vessel type to Faster Indicators simplified reporting vessel type mappings: 70
2026-03-10 11:02:29,912 - ungp_port_calls - INFO - UN Locode to Faster Indicators simplified reporting (super port) locode mappings: 47
2026-03-10 11:02:29,926 - ungp_port_calls - INFO - Faster Indicators reporting port zone count: 16


### Prepare data (pandas)

In [16]:
fi_port_calls_df = (
    port_calls_sdf
    .filter(col('locode').startswith(country_code))
    .filter(col('in_port_zone') == True)
    .filter(~col('imo').isin(excluded_imos))
    .filter(col("time_arrival") > reporting_start_date)
    .limit(100000)
    .toPandas()
    .pipe(reporting.time_arrival_to_dates)
    .merge(fi_ship_types_map_df[['StatCode5', 'fi_vessel_type']], on='StatCode5', how='left')
    .pipe(reporting.add_reporting_ports, fi_locode_map_df)
    .query(f'reporting_locode in {fi_port_locodes}')
    .reset_index(drop=True)
)
ct_fi_port_calls = len(fi_port_calls_df)
logger.info(f'Faster indicators port visit count: {ct_fi_port_calls}')

t_pandasout_done = pd.Timestamp.now()

fi_port_calls_df.head()

2026-03-10 11:02:33,056 - ungp_port_calls - INFO - Faster indicators port visit count: 1489


,imo,mmsi,ShipName,ShiptypeLevel5,StatCode5,locode,port_name,time_arrival,time_departure,in_port_zone,subvisit_count,time_static_total,time_in_port,GrossTonnage,NetTonnage,TEU,date_arrival,week_arrival,fi_vessel_type,reporting_locode
0,9268851,255802870,FL STOROE,General Cargo Ship,A31A2GX,GBTIL,Tilbury,2025-12-02 22:24:05,2025-12-14 02:19:31,True,1,267.923889,267.923889,3995,1559,252,2025-12-02,2025-12-01,Cargo,GBLON
1,9274848,219435000,FREESIA SEAWAYS,Ro-Ro Cargo Ship,A35A2RR,GBIMM,Immingham,2025-12-13 08:51:38,NaT,True,1,38.986944,38.986944,37939,11381,90,2025-12-13,2025-12-08,Cargo,GBGSY
2,9274848,219435000,FREESIA SEAWAYS,Ro-Ro Cargo Ship,A35A2RR,GBIMM,Immingham,2025-12-06 07:49:35,2025-12-06 17:02:55,True,1,9.222222,9.222222,37939,11381,90,2025-12-06,2025-12-01,Cargo,GBGSY
3,9454759,636014655,BALTIC KLIPPER,Refrigerated Cargo Ship,A34A2GR,GBPME,Portsmouth,2025-12-08 22:27:41,2025-12-13 18:30:59,True,1,116.055000,116.055000,14091,7238,552,2025-12-08,2025-12-08,Cargo,GBPME
4,9455715,311048300,RCC PRESTIGE,Vehicles Carrier,A35B2RV,GBTYN,Tyne,2025-12-13 14:44:36,2025-12-14 07:17:07,True,1,16.541944,16.541944,36834,11050,0,2025-12-13,2025-12-08,Carrier,GBTYN


### Aggregated reports

In [17]:
report_plan = '''
title,filename,vessel_types,arrival_key,location_key,aggfield,aggfunc
Daily all visits,daily_all_visits.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",date_arrival,reporting_locode,imo,count
Daily C&T visits,daily_cargotanker_visits.csv,"['Cargo', 'Carrier', 'Tanker']",date_arrival,reporting_locode,imo,count
Daily all unique ships,daily_all_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",date_arrival,reporting_locode,imo,nunique
Daily C&T unique ships,daily_cargotanker_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker']",date_arrival,reporting_locode,imo,nunique
Weekly all visits,weekly_all_visits.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",week_arrival,reporting_locode,imo,count
Weekly C&T visits,weekly_cargotanker_visits.csv,"['Cargo', 'Carrier', 'Tanker']",week_arrival,reporting_locode,imo,count
Weekly all unique ships,weekly_all_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",week_arrival,reporting_locode,imo,nunique
Weekly C&T unique ships,weekly_cargotanker_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker']",week_arrival,reporting_locode,imo,nunique
Passenger Daily Visits,daily_passenger_visits.csv,['Passenger'],date_arrival,reporting_locode,imo,count
Tanker Weekly Gross Tonnage,tanker_weekly_grosstonnage.csv,['Tanker'],week_arrival,reporting_locode,GrossTonnage,sum
Cargo Weekly TEUs,cargo_weekly_teu.csv,['Cargo'],week_arrival,reporting_locode,TEU,sum
Cargo Weekly Mean Time in Port,cargo_weekly_meantime.csv,['Cargo'],week_arrival,reporting_locode,time_in_port,mean
'''

report_plan_df = pd.read_csv(StringIO(report_plan))[['title', 'filename', 'vessel_types', 'arrival_key', 'location_key', 'aggfield', 'aggfunc']]

reports_df = (
    report_plan_df
    .drop(columns=['location_key', 'aggfield', 'aggfunc'])
    .assign(link = report_plan_df.apply(lambda x: reporting.make_report(x, fi_port_calls_df, fi_mangle=mangle, output_link=True), axis='columns'))
)

HTML(reports_df.to_html(render_links=True, escape=False))

,title,filename,vessel_types,arrival_key,link
0,Daily all visits,daily_all_visits.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",date_arrival,"<a download=""daily_all_visits.csv"" href=""data:text/csv;base64,ZGF0ZV9hcnJpdmFsLEdCQkVMLEdCRFZSLEdCRk9SLEdCRlhULEdCR1NZLEdCSExZLEdCSFVMLEdCTEFSLEdCTElWLEdCTE9OLEdCTUxGLEdCUE1FLEdCU09VLEdCVEVFLEdCVFlOLEdCV1BULEFsbCBQb3J0cwoyMDI1LTEyLTAxLDE0LDMxLDIsNCwxMCw1LDYsNCw4LDExLDAsNCwzLDksMSwxLDExMwoyMDI1LTEyLTAyLDksMzcsMSw3LDYsNiwyLDYsOCwxNCwzLDMsNiwyLDEsMCwxMTEKMjAyNS0xMi0wMywxMSwzNywwLDYsOSw2LDgsNiw0LDEwLDQsMiw2LDUsMywwLDExNwoyMDI1LTEyLTA0LDEzLDQyLDQsOCwxMSw1LDYsNSwyLDQsMiw0LDYsNiwxLDIsMTIxCjIwMjUtMTItMDUsOCwzNywyLDQsMTgsNCwzLDUsMywxMiwzLDMsNCw1LDEsMywxMTUKMjAyNS0xMi0wNiwxMiwzNiwwLDQsMTMsNiwxLDcsNCw2LDEsMywzLDYsMiwxLDEwNQoyMDI1LTEyLTA3LDUsMzAsMywzLDYsNCwyLDQsOCwxMCwyLDYsMSwxLDAsMCw4NQoyMDI1LTEyLTA4LDcsMzUsMiw3LDgsNyw0LDUsNiw1LDQsNSwyLDMsMSwwLDEwMQoyMDI1LTEyLTA5LDgsMzYsMSw2LDgsMCwzLDIsNSwxMiwwLDUsNCwzLDEsMCw5NAoyMDI1LTEyLTEwLDYsMzUsMSw2LDEwLDQsNSw0LDQsOCwzLDMsNSw1LDEsMCwxMDAKMjAyNS0xMi0xMSw3LDM3LDIsOCwxMCw0LDIsNSw4LDExLDMsOCw2LDMsMSwwLDExNQoyMDI1LTEyLTEyLDEwLDM1LDEsNyw5LDUsMiw3LDEwLDEwLDQsMiw1LDUsMSwwLDExMwoyMDI1LTEyLTEzLDExLDM1LDIsNSw5LDUsMiw1LDcsOSwyLDMsNiw3LDMsMSwxMTIKMjAyNS0xMi0xNCw4LDMzLDAsMyw4LDQsMCwzLDMsMTAsMiw0LDYsMiwxLDAsODcK"" target=""_blank"">Daily all visits (daily_all_visits.csv)"
1,Daily C&T visits,daily_cargotanker_visits.csv,"['Cargo', 'Carrier', 'Tanker']",date_arrival,"<a download=""daily_cargotanker_visits.csv"" href=""data:text/csv;base64,ZGF0ZV9hcnJpdmFsLEdCQkVMLEdCRFZSLEdCRk9SLEdCRlhULEdCR1NZLEdCSFVMLEdCTEFSLEdCTElWLEdCTE9OLEdCTUxGLEdCUE1FLEdCU09VLEdCVEVFLEdCVFlOLEdCV1BULEFsbCBQb3J0cwoyMDI1LTEyLTAxLDYsMCwyLDIsOSw1LDAsNywxMSwwLDAsMyw5LDAsMSw1NQoyMDI1LTEyLTAyLDMsMiwxLDUsNiwyLDAsNywxMywzLDEsNSwyLDEsMCw1MQoyMDI1LTEyLTAzLDQsMCwwLDQsOCw4LDAsNCw5LDIsMCw0LDUsMiwwLDUwCjIwMjUtMTItMDQsNSwxLDQsNiwxMCw2LDAsMiwyLDAsMSw1LDYsMCwyLDUwCjIwMjUtMTItMDUsMiwwLDIsMiwxNywzLDAsMywxMiwzLDEsMiw1LDAsMyw1NQoyMDI1LTEyLTA2LDQsMCwwLDMsMTIsMSwxLDMsNiwxLDEsMiw2LDEsMSw0MgoyMDI1LTEyLTA3LDAsMCwzLDIsNSwyLDAsNiwxMCwyLDEsMCwxLDAsMCwzMgoyMDI1LTEyLTA4LDEsMSwyLDUsNyw0LDAsNiw1LDIsMiwyLDMsMSwwLDQxCjIwMjUtMTItMDksMywxLDEsNCw4LDMsMCw0LDEyLDAsMCw0LDMsMCwwLDQzCjIwMjUtMTItMTAsMiwwLDEsNCw5LDQsMCwzLDYsMiwxLDQsNSwwLDAsNDEKMjAyNS0xMi0xMSwzLDAsMiw2LDksMiwwLDcsMTAsMywxLDUsMywwLDAsNTEKMjAyNS0xMi0xMiwzLDAsMSw1LDgsMiwxLDUsMTAsMywwLDMsNSwwLDAsNDYKMjAyNS0xMi0xMyw2LDAsMiwzLDksMiwwLDYsOSwxLDEsNSw3LDIsMSw1NAoyMDI1LTEyLTE0LDEsMCwwLDEsNywwLDAsMiwxMCwxLDIsNSwyLDAsMCwzMQo="" target=""_blank"">Daily C&T visits (daily_cargotanker_visits.csv)"
2,Daily all unique ships,daily_all_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker', 'Passenger']",date_arrival,"<a download=""daily_all_unique_ships.csv"" href=""data:text/csv;base64,ZGF0ZV9hcnJpdmFsLEdCQkVMLEdCRFZSLEdCRk9SLEdCRlhULEdCR1NZLEdCSExZLEdCSFVMLEdCTEFSLEdCTElWLEdCTE9OLEdCTUxGLEdCUE1FLEdCU09VLEdCVEVFLEdCVFlOLEdCV1BULEFsbCBQb3J0cwoyMDI1LTEyLTAxLDEwLDksMiw0LDEwLDQsNiwyLDgsMTEsMCw0LDMsNywxLDEsODEKMjAyNS0xMi0wMiw2LDEyLDEsNyw2LDQsMiwyLDYsMTQsMywyLDYsMiwxLDAsNzMKMjAyNS0xMi0wMyw3LDEwLDAsNiw4LDQsOCwyLDQsMTAsMywyLDYsNSwzLDAsNzYKMjAyNS0xMi0wNCw5LDEyLDQsOCwxMSwzLDUsMiwyLDQsMSw0LDUsNiwxLDIsNzgKMjAyNS0xMi0wNSw1LDExLDIsNCwxOCwzLDMsMiwzLDEyLDMsMyw0LDUsMSwzLDgyCjIwMjUtMTItMDYsNywxMSwwLDQsMTIsMywxLDMsMyw2LDEsMywzLDYsMiwxLDY0CjIwMjUtMTItMDcsMywxMSwyLDMsNiwzLDIsMiw3LDEwLDIsNSwxLDEsMCwwLDU4CjIwMjUtMTItMDgsNSwxMiwyLDcsOCw0LDQsMiw1LDUsMyw1LDIsMywxLDAsNjgKMjAyNS0xMi0wOSw2LDEyLDEsNiw4LDAsMywyLDUsMTIsMCw0LDQsMywxLDAsNjcKMjAyNS0xMi0xMCw1LDExLDEsNiwxMCwzLDUsMiw0LDgsMywzLDUsNSwxLDAsNjkKMjAyNS0xMi0xMSw2LDExLDIsOCwxMCwzLDIsMiw3LDExLDIsNiw2LDMsMSwwLDc5CjIwMjUtMTItMTIsNiwxMSwxLDcsOSw0LDIsMyw5LDEwLDQsMiw1LDQsMSwwLDc2CjIwMjUtMTItMTMsOSwxMSwyLDUsOSwzLDIsMiw3LDksMiwzLDYsNiwzLDEsNzkKMjAyNS0xMi0xNCw1LDExLDAsMyw4LDMsMCwyLDMsOSwyLDQsNiwyLDEsMCw1OQo="" target=""_blank"">Daily all unique ships (daily_all_unique_ships.csv)"
3,Daily C&T unique ships,daily_cargotanker_unique_ships.csv,"['Cargo', 'Carrier', 'Tanker']",date_arrival,"<

### Traffic

In [18]:
visits_by_port_vessel_daily_df = reporting.visits_by_port_vessel(fi_port_calls_df)
display(utils.create_download_link(df=visits_by_port_vessel_daily_df, title='All Daily Traffic', filename='all_daily_traffic_mmsis'))
display(visits_by_port_vessel_daily_df.head())

visits_by_port_vessel_weekly_df = reporting.visits_by_port_vessel(fi_port_calls_df, arrival_key='week_arrival')
display(utils.create_download_link(df=visits_by_port_vessel_weekly_df, title='All Weekly Traffic', filename='all_weekly_traffic_mmsis'))
display(visits_by_port_vessel_weekly_df.head())

,date_arrival,reporting_locode,imo,mmsi,visits,ShipName,fi_vessel_type,ShiptypeLevel5,GrossTonnage,NetTonnage,TEU
0,2025-12-01,GBBEL,8709767,275486000,1,CEG UNIVERSE,Cargo,General Cargo Ship,852,490,0
1,2025-12-01,GBBEL,9100126,308218000,1,SWAMI,Cargo,General Cargo Ship,2839,1609,194
2,2025-12-01,GBBEL,9121625,232018002,1,STENA SCOTIA,Cargo,Ro-Ro Cargo Ship,13017,3905,0
3,2025-12-01,GBBEL,9198941,235089435,2,STENA SUPERFAST VII,Passenger,Passenger/Ro-Ro Ship (Vehicles),30285,10769,0
4,2025-12-01,GBBEL,9198953,235089436,4,STENA SUPERFAST VIII,Passenger,Passenger/Ro-Ro Ship (Vehicles),30285,10769,0


,week_arrival,reporting_locode,imo,mmsi,visits,ShipName,fi_vessel_type,ShiptypeLevel5,GrossTonnage,NetTonnage,TEU
0,2025-12-01,GBBEL,8709767,275486000,1,CEG UNIVERSE,Cargo,General Cargo Ship,852,490,0
1,2025-12-01,GBBEL,8806163,277334000,1,ARINA,Cargo,General Cargo Ship,3826,2043,326
2,2025-12-01,GBBEL,9100126,308218000,1,SWAMI,Cargo,General Cargo Ship,2839,1609,194
3,2025-12-01,GBBEL,9121625,232018002,4,STENA SCOTIA,Cargo,Ro-Ro Cargo Ship,13017,3905,0
4,2025-12-01,GBBEL,9144263,259538000,1,LYSBRIS SEAWAYS,Cargo,General Cargo Ship,7409,4568,160


### Checks
Collect sanity checks of outputs to diagnose errors during the execution

In [19]:
dic_checks = {
	'Port calls': ct_port_calls,
	'Active vessels': ct_vessels_active,
	'Port calls by active vessels': ct_port_calls_by_active_vessels,
	'Vessels active in UK': ct_vessels_active_gb,
	'Excluded IMOs': ct_excluded_imos,
	'FI Ship types map': ct_fi_ship_types_map,
	'FI LOCODE map': ct_fi_locode_map,
	'FI port LOCODEs': ct_fi_port_locodes,
	'FI port calls': ct_fi_port_calls
}

dic_times = {
	'Calculation start': t_calc_start,
	'AIS df complete': t_aisdf_done,
	'Port calls complete': t_portcalls_done,
	'Pandas DataFrame generated': t_pandasout_done
}

In [20]:
df_checks = livechecks.sanity_check(dic_checks)
df_checks.head(10)

,Reference,Reported,Difference (%),Expected
Port calls,184600.0,125419.0,-32.05905,True
Active vessels,1680.0,1306.0,-22.26190,True
Port calls by active vessels,32700.0,22111.0,-32.38226,True
Vessels active in UK,1580.0,1177.0,-25.50633,True
Excluded IMOs,50.0,36.0,-28.00000,True
FI Ship types map,70.0,70.0,0.00000,True
FI LOCODE map,47.0,47.0,0.00000,True
FI port LOCODEs,16.0,16.0,0.00000,True
FI port calls,2650.0,1489.0,-43.81132,True


In [21]:
df_times = livechecks.time_check(dic_times)
df_times.head()

,Calculation start,AIS df complete,Port calls complete,Pandas DataFrame generated,Total processing time
0,2026-03-10 11:00:08.478084,2026-03-10 11:00:14.199814,2026-03-10 11:00:57.005652,2026-03-10 11:02:33.057451,0 days 00:02:24.579367


In [22]:
# output sanity checks to csv
livechecks.create_download_link(df_checks, 'Download sanity checks', 'sanity_checks.csv')

In [23]:
# output time checks to csv
livechecks.create_download_link(df_times, 'Download time checks', 'time_checks.csv')